# 🕵️ Dia 24 — Enumeração de username e mapeamento da pegada digital (a sua)

**Investigador Cyber 90 Dias** · Fase 2 · Rank Jutsu `C`

> Baseado no *Day 24* de [cyber-investigator-skills](https://github.com/thiefdroppers/cyber-investigator-skills/blob/main/curriculum/phase-2-osint-digital-footprint/day-24.md), © ThiefDroppers (MIT).
> Notebook, camada BR e Camada 8 por Deivison Lourenço ([@Ridd1kulusC0d3r](https://github.com/Ridd1kulusC0d3r)).

---

### 🎬 Onde paramos

Um cooperado denunciou o perfil `@serraazul.oficial`. Você ia rodar o Sherlock em "pesc". Rafa segurou seu pulso: *"E se o 'pesc' for um aposentado de Lavras? Hoje você roda em você mesmo, onde sabe a resposta, e mede."*

> ⚠️ **Regra do notebook:** tudo aqui roda sobre **dados sintéticos**, sobre um **servidor de brinquedo na sua própria máquina** ou sobre **handles seus**. Nenhuma célula consulta handles de terceiros.

Neste notebook você vai:

1. **Ver o falso positivo nascer** num servidor HTTP local e escrever um checador melhor.
2. **Jogar o quiz do fluxograma**.
3. **Medir a precisão** das três ferramentas na marca da cooperativa.
4. **Preencher a sua tabela** e gerar o grafo da sua pegada em Mermaid.

🐍 Só biblioteca padrão do Python. Nada para instalar. Rode as células de cima para baixo (Shift+Enter). Tudo o que é do caso é **fictício** (domínios `.example`, IPs de documentação).

## 1. Como nasce um falso positivo

A célula sobe um servidor HTTP **só na sua máquina** (`127.0.0.1`) com três "sites" de brinquedo:

- `/honesto/<user>`: 200 se o usuário existe, 404 se não;
- `/mentiroso/<user>`: **sempre 200**, com o texto "usuário não encontrado" quando não existe;
- `/bloqueio/<user>`: sempre 429 ("too many requests"), como um site que bloqueia robôs.

Se o ambiente não deixar abrir porta, a célula simula as respostas sem rede.

In [ ]:
import threading
import urllib.request, urllib.error
from http.server import BaseHTTPRequestHandler, HTTPServer

USUARIOS = {"meuhandle"}                          # o único usuário que "existe" nos sites de brinquedo

class Brinquedo(BaseHTTPRequestHandler):
    def do_GET(self):
        partes = self.path.strip("/").split("/")  # ex.: ["mentiroso", "fulano"]
        site, user = (partes + [""])[:2]
        existe = user in USUARIOS
        if site == "honesto":
            codigo, corpo = (200, f"Perfil de {user}") if existe else (404, "Not found")
        elif site == "mentiroso":
            codigo, corpo = 200, (f"Perfil de {user}" if existe else "Ops! Usuário não encontrado.")
        else:                                     # bloqueio
            codigo, corpo = 429, "Too many requests"
        self.send_response(codigo)
        self.send_header("Content-Type", "text/plain; charset=utf-8")
        self.end_headers()
        self.wfile.write(corpo.encode("utf-8"))
    def log_message(self, *args):                 # silencia o log do servidor
        pass

try:
    servidor = HTTPServer(("127.0.0.1", 0), Brinquedo)    # porta 0 = o sistema escolhe uma livre
    PORTA = servidor.server_address[1]
    threading.Thread(target=servidor.serve_forever, daemon=True).start()
    BASE = f"http://127.0.0.1:{PORTA}"
    print("Servidor de brinquedo em", BASE)
except OSError as erro:
    BASE = None
    print(f"(Sem porta local: {erro}. Vou simular as respostas.)")

def buscar(site, user):
    """Devolve (código HTTP, corpo). Usa o servidor, ou simula se ele não subiu."""
    if BASE is None:
        existe = user in USUARIOS
        tabela = {"honesto": (200, "Perfil") if existe else (404, "Not found"),
                  "mentiroso": (200, "Perfil" if existe else "Ops! Usuário não encontrado."),
                  "bloqueio": (429, "Too many requests")}
        return tabela[site]
    try:
        with urllib.request.urlopen(f"{BASE}/{site}/{user}", timeout=5) as r:
            return r.status, r.read().decode("utf-8")
    except urllib.error.HTTPError as e:
        return e.code, e.read().decode("utf-8")

In [ ]:
def checador_ingenuo(site, user):
    """O que muita ferramenta faz: 200 = existe."""
    codigo, _ = buscar(site, user)
    return codigo == 200

def checador_melhor(site, user, texto_nao_existe="não encontrado"):
    """Lê o corpo. Três respostas: True, False ou None (inconclusivo)."""
    codigo, corpo = buscar(site, user)
    if codigo == 404:
        return False
    if codigo != 200:
        return None                                # 429, 403...: o site bloqueou, verifique à mão
    return texto_nao_existe not in corpo           # 200 com "não encontrado" = não existe

print(f"{'site':<11}{'usuário':<14}{'ingênuo':<10}{'melhor'}")
for site in ("honesto", "mentiroso", "bloqueio"):
    for user in ("meuhandle", "naoexiste123"):
        print(f"{site:<11}{user:<14}{str(checador_ingenuo(site, user)):<10}{checador_melhor(site, user)}")
print("\n'mentiroso' + 'naoexiste123' = o falso positivo do Pinterest no original.")
print("'bloqueio' + 'meuhandle' = o 'erro' em um site que você SABE que usa: verificar à mão.")

## 2. 🎮 O fluxograma

1 = falso positivo · 2 = mesmo handle, dono não provado · 3 = verificado · 4 = verificar à mão (provável bloqueio)

In [ ]:
QUIZ = [
    ("A ferramenta diz 'acerto'. Você abre: 'Ops! Usuário não encontrado.' (HTTP 200)", "1",
     "Página genérica servida com 200. Fica na tabela, fora do grafo."),
    ("Acerto no Chess.com: perfil real, entrou em 2012, outro país, avatar diferente", "2",
     "Perfil real, sem segundo atributo. Outra pessoa (ou dono não provado)."),
    ("Acerto no GitHub: perfil real, com os seus repositórios e o seu avatar", "3",
     "Segundo atributo independente (avatar + repositórios). Entra no grafo."),
    ("'Erro' num fórum onde você TEM conta desde 2011", "4",
     "Provavelmente o site bloqueia checagem automatizada. Verifique à mão."),
    ("Acerto 'serraazul' no TikTok: loja de tintas de outro estado, não está na lista oficial", "2",
     "Mesmo handle, não oficial. Não investigue quem é. Vai para a lista da Marta, se ela quiser."),
]
MODO_JOGO = True
acertos = 0
for caso, certo, porque in QUIZ:
    print(f"\n• {caso}")
    if MODO_JOGO:
        r = ""
        while r not in ("1", "2", "3", "4"):
            r = input("   1 falso positivo / 2 dono não provado / 3 verificado / 4 verificar à mão: ").strip()
        acertos += (r == certo)
        print("   ✅" if r == certo else "   ❌", end="")
    print(f"   {porque}")
if MODO_JOGO:
    print(f"\n🏁 {acertos}/{len(QUIZ)}")

## 3. Precisão na marca da cooperativa (dados sintéticos, autorizados)

Mesmos dados de [`caso-continuo/dados/dia-24/`](https://github.com/Ridd1kulusC0d3r/investigador-cyber-90-dias/tree/main/caso-continuo/dados/dia-24). Precisão = acertos verdadeiros (oficial verificado) ÷ todos os acertos relatados.

In [ ]:
# Lista oficial (página Canais oficiais, Dia 19): (plataforma, handle, url)
OFICIAIS = [
 [
  "Instagram",
  "serraazul",
  "https://instagram.example/serraazul"
 ],
 [
  "YouTube",
  "serraazulcoop",
  "https://youtube.example/@serraazulcoop"
 ],
 [
  "Facebook",
  "serraazulcoop",
  "https://facebook.example/serraazulcoop"
 ],
 [
  "LinkedIn",
  "cooperativa-serra-azul",
  "https://linkedin.example/company/cooperativa-serra-azul"
 ]
]
# Verificação manual: (site, handle, categoria, evidência)
VERDADE = [
 [
  "Instagram",
  "serraazul",
  "oficial",
  "Na lista oficial; bio linka serraazul.example; mesmo logotipo"
 ],
 [
  "YouTube",
  "serraazulcoop",
  "oficial",
  "Na lista oficial; descrição linka serraazul.example"
 ],
 [
  "Facebook",
  "serraazulcoop",
  "oficial",
  "Na lista oficial; mesmo CNPJ fictício 00.000.000/0001-00 na página"
 ],
 [
  "Pinterest",
  "serraazul",
  "falso_positivo",
  "Página genérica 'usuário não encontrado' com HTTP 200"
 ],
 [
  "Pinterest",
  "serraazulcoop",
  "falso_positivo",
  "Página genérica 'usuário não encontrado' com HTTP 200"
 ],
 [
  "TikTok",
  "serraazul",
  "mesmo_handle_nao_oficial",
  "Perfil real de loja de tintas 'Serra Azul' em outro estado; não está na lista"
 ],
 [
  "Etsy",
  "serraazul",
  "mesmo_handle_nao_oficial",
  "Perfil real de artesanato; não está na lista"
 ],
 [
  "Instagram",
  "serraazul.oficial",
  "mesmo_handle_nao_oficial",
  "NÃO está na lista oficial; usa o logotipo; oferece 'desbloqueio de Pix' (ver Dia 25)"
 ],
 [
  "Telegram",
  "serraazulcoop",
  "falso_positivo",
  "Telegram responde 200 com página de convite genérica"
 ],
 [
  "Kwai",
  "serraazul",
  "falso_positivo",
  "Redireciona para a página inicial com HTTP 200"
 ]
]
# O que cada ferramenta relatou como acerto
RELATOS = {
 "Sherlock": [
  [
   "Instagram",
   "serraazul"
  ],
  [
   "YouTube",
   "serraazulcoop"
  ],
  [
   "Pinterest",
   "serraazul"
  ],
  [
   "Pinterest",
   "serraazulcoop"
  ],
  [
   "TikTok",
   "serraazul"
  ],
  [
   "Telegram",
   "serraazulcoop"
  ],
  [
   "Kwai",
   "serraazul"
  ]
 ],
 "Maigret": [
  [
   "Instagram",
   "serraazul"
  ],
  [
   "YouTube",
   "serraazulcoop"
  ],
  [
   "Facebook",
   "serraazulcoop"
  ],
  [
   "TikTok",
   "serraazul"
  ],
  [
   "Etsy",
   "serraazul"
  ],
  [
   "Telegram",
   "serraazulcoop"
  ]
 ],
 "WhatsMyName": [
  [
   "Instagram",
   "serraazul"
  ],
  [
   "Facebook",
   "serraazulcoop"
  ],
  [
   "Etsy",
   "serraazul"
  ]
 ]
}

cat = {(s, h): c for s, h, c, _e in VERDADE}
print(f"{'Ferramenta':<13}{'Acertos':>8}{'Oficiais':>9}{'Precisão':>10}")
for ferramenta, lista in RELATOS.items():
    verdadeiros = sum(1 for k in lista if cat.get(tuple(k)) == "oficial")
    print(f"{ferramenta:<13}{len(lista):>8}{verdadeiros:>9}{verdadeiros / len(lista):>10.0%}")

print("\nLista para a Marta — uso do nome NÃO listado nos canais oficiais (não investigado):")
for s, h, c, e in VERDADE:
    if c == "mesmo_handle_nao_oficial":
        print(f"  {s}: @{h} — {e}")

> Repare que nenhuma ferramenta achou o `@serraazul.oficial`: ele veio de um cooperado. Enumeração de handle procura o handle **exato**; imitadores usam variações. Variações de marca são o tema do Dia 25 (pelo lado da plataforma) e, no futuro, de ferramentas como o radar de imitadores.

## 4. 📝 A sua tabela e o grafo da sua pegada

Preencha com os resultados da **sua** execução (Passos 2 a 6). Uma linha por site, no formato `site;sherlock;maigret;wmn;é_meu;evidência;aresta`, onde `aresta` é o atributo que liga a conta (`mesmo handle`, `mesmo e-mail`, `perfil linka`, `mesmo avatar`) ou vazio se não for sua.

A célula só roda se você marcar a trava `confirmo_que_sao_meus`.

In [ ]:
#@title Minha tabela de verificação { display-mode: "form" }
confirmo_que_sao_meus = True  #@param {type:"boolean"}
meus_handles = "meuhandle1, meuhandle2"  #@param {type:"string"}
meu_email = "eu@mail.example"  #@param {type:"string"}
linhas = "GitHub;acerto;acerto;acerto;sim;meus repositórios;mesmo handle | Chess.com;acerto;acerto;erro;não;outra pessoa, país diferente; | Pinterest;acerto;erro;erro;não há perfil;200 com 'não encontrado' (falso positivo); | Fórum antigo;erro;erro;acerto;sim, esquecido;meus posts de 2011;mesmo avatar"  #@param {type:"string"}

assert confirmo_que_sao_meus, "Este laboratório é só sobre a SUA pegada. Marque a trava para continuar."

registros = []
for bruto in linhas.split("|"):
    campos = [c.strip() for c in bruto.split(";")]
    campos += [""] * (7 - len(campos))                    # completa campos vazios
    registros.append(dict(zip(["site", "sherlock", "maigret", "wmn", "meu", "evid", "aresta"], campos)))

print("| Site | Sherlock | Maigret | WMN | É meu? | Evidência |\n|---|---|---|---|---|---|")
for r in registros:
    print(f"| {r['site']} | {r['sherlock']} | {r['maigret']} | {r['wmn']} | {r['meu']} | {r['evid']} |")

# Precisão (Passo 9): acertos verificados como meus / todos os acertos relatados
print()
for ferramenta in ("sherlock", "maigret", "wmn"):
    hits = [r for r in registros if r[ferramenta] == "acerto"]
    meus = [r for r in hits if r["meu"].startswith("sim")]
    print(f"Precisão {ferramenta:<9}: {len(meus)}/{len(hits)}" + (f" = {len(meus)/len(hits):.0%}" if hits else ""))

# Grafo Mermaid (Passo 7): só contas verificadas como minhas; falsos positivos ficam de fora
handles = [h.strip() for h in meus_handles.split(",") if h.strip()]
mermaid = ["graph LR"]
for i, h in enumerate(handles):
    mermaid.append(f'    H{i}(["handle<br/>{h}"])')
mermaid.append(f'    EM(["e-mail<br/>{meu_email}"])')
for j, r in enumerate(registros):
    if r["meu"].startswith("sim") and r["aresta"]:
        origem = "EM" if "e-mail" in r["aresta"] else "H0"
        mermaid.append(f'    {origem} -->|"{r["aresta"]}"| C{j}["{r["site"]}"]')
print("\n```mermaid\n" + "\n".join(mermaid) + "\n```")
print("\n(Cole no README do seu fork. Ajuste a origem de cada aresta para o handle certo.)")

### 🧠 Camada 8

Cinco acertos para "pesc" *parecem* a mesma pessoa. É a **heurística da representatividade**, ignorando a **taxa de base**: quantas pessoas escolheriam aquele handle? Quanto mais comum o handle, mais atributos independentes você exige.

## ✅ Antes de fechar o Dia 24

- [ ] Você viu o falso positivo nascer e o checador melhor devolver `False` / `None`
- [ ] Três números de precisão da sua pegada **e** três da marca da cooperativa
- [ ] Tabela com pelo menos um falso positivo e o motivo
- [ ] Grafo da sua pegada só com contas verificadas, arestas rotuladas
- [ ] Nas notas: "nenhuma ferramenta foi rodada sobre handle de terceiro"

**Amanhã (Dia 25):** redes sociais pela própria plataforma. O `@serraazul.oficial` tem data de criação, histórico de nome e anúncios. Você vai ler, sem seguir, curtir nem comentar.

---
<sub>Baseado em cyber-investigator-skills por ThiefDroppers (MIT) · Investigador Cyber 90 Dias por Deivison Lourenço</sub>